In [ ]:
import pprint

from swsearch.config import settings
from swsearch.extract.wikidump import convert_json_array_to_jsonl, save_article_titles, traverse_directory
from swsearch.embed.paragraphs import embed_paragraphs
from swsearch.index.faiss_store import build_flat_index_from_manifest
from swsearch.metadata.store import load_faiss_meta_sqlite
from swsearch.search.engine import SearchEngine

paths = settings.paths

In [ ]:
# Equivalent to `swsearch extract`. Skips if already extracted -- this demo
# reuses the existing extracted corpus rather than reprocessing it.
if not paths.jsonl_dir.exists():
    traverse_directory(str(paths.extracted_dir), str(paths.json_dir))
    convert_json_array_to_jsonl(str(paths.json_dir), str(paths.jsonl_dir))
    save_article_titles(str(paths.jsonl_dir), str(paths.article_titles_path))
else:
    print(f"{paths.jsonl_dir} already exists, skipping extract.")

In [ ]:
# Equivalent to `swsearch embed`. Splits articles into paragraphs, embeds them,
# and writes metadata + a manifest to the SQLite store alongside the .npy
# batches, so build-index below can never drift out of alignment with them.
if not paths.faiss_meta_db_path.exists():
    embed_paragraphs(
        data_dir=str(paths.json_dir),
        embeddings_dir=str(paths.embeddings_dir),
        meta_db_path=str(paths.faiss_meta_db_path),
        model_name=settings.model.embedding_model_name,
        device=settings.model.device,
    )
else:
    print(f"{paths.faiss_meta_db_path} already exists, skipping embed.")

In [ ]:
# Equivalent to `swsearch build-index`. Builds the FAISS index by replaying
# the metadata manifest (not sorted(glob(...))), then asserts index.ntotal
# matches the meta DB row count before writing it out.
if not paths.faiss_index_path.exists():
    meta_conn = load_faiss_meta_sqlite(str(paths.faiss_meta_db_path))
    build_flat_index_from_manifest(str(paths.embeddings_dir), meta_conn, str(paths.faiss_index_path))
else:
    print(f"{paths.faiss_index_path} already exists, skipping build-index.")

In [ ]:
# Equivalent to `swsearch search "<query>"`. SearchEngine embeds the query,
# retrieves FAISS candidates, reranks by cosine similarity, and deduplicates
# to one best-scoring result per article -- the recipe this project started
# from, now backed by the paragraph-level index shared with triplet mining.
engine = SearchEngine()
results = engine.search("Who invented science?", k=5)

pprint.pprint([{"title": r["title"], "url": r["url"], "score": round(r["score"], 4)} for r in results])